# Lab 3: Word Embeddings (Word2Vec & GloVe)

**Objective:** In this lab, you will train a Word2Vec model on a custom text corpus, explore cosine similarities, visualize the high-dimensional word vectors in 2D space using t-SNE, and compare your custom embeddings with pre-trained GloVe embeddings.

**Estimated Time:** 2 Hours
**Difficulty:** Easy to Medium

### Outline:
1. **Task 1:** Corpus Preparation & Preprocessing
2. **Task 2:** Training a Word2Vec Model
3. **Task 3:** Measuring Cosine Similarity
4. **Task 4:** Visualizing Embeddings with t-SNE
5. **Task 5:** Comparing with Pre-trained GloVe Embeddings

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import gensim
import gensim.downloader as api
import re


raw_corpus = [
    "India is a beautiful country located in South Asia.",
    "New Delhi is the capital of India.",
    "Mumbai is the financial capital of India and a very busy city.",
    "India has a rich culture and a very diverse heritage.",
    "Many vibrant festivals like Diwali and Holi are celebrated in India.",
    "The Himalayas are a beautiful mountain range in northern India.",
    "Cricket is a very popular sport in India.",
    "The Taj Mahal is a beautiful monument located in Agra, India.",
    "Spicy food and tea are very popular across the country."
]

## Task 1: Corpus Preparation & Preprocessing (Easy)

Word2Vec in `gensim` expects a list of tokenized sentences (a list of lists of words). We need to clean our text by removing punctuation, lowercasing everything, and splitting the sentences into words.

**Your Task:** Write a function to preprocess the `raw_corpus` into a list of tokenized sentences.

In [ ]:
def preprocess_corpus(corpus):
    tokenized_corpus = []
    for sentence in corpus:
        
        sentence = sentence.lower()
        # Remove punctuation using regex
        sentence = re.sub(r'[^\w\s]', '', sentence)
        
        tokens = sentence.split()
        tokenized_corpus.append(tokens)
    return tokenized_corpus

# Test preprocessing
processed_corpus = preprocess_corpus(raw_corpus)
print("First sentence tokenized:", processed_corpus[0])

## Task 2: Training a Word2Vec Model (Medium)

Now we will use `gensim.models.Word2Vec` to train our own word embeddings.

Key parameters you will use:
* `sentences`: Your tokenized corpus.
* `vector_size`: The dimensionality of the word vectors (e.g., 20 for this small corpus).
* `window`: The maximum distance between the current and predicted word within a sentence (e.g., 2).
* `min_count`: Ignores words with total frequency lower than this (Set to 1 so we don't lose words in our toy corpus).

**Your Task:** Initialize and train the Word2Vec model.

In [ ]:

w2v_model = gensim.models.Word2Vec(
    sentences=processed_corpus, 
    vector_size=20, 
    window=2, 
    min_count=1, 
    epochs=50 
)

print("Model trained successfully!")
print(f"Vocabulary size: {len(w2v_model.wv.index_to_key)}")


print("\nVector for 'india':\n", w2v_model.wv['india'])

## Task 3: Measuring Cosine Similarity (Easy)

Word embeddings capture semantic meaning. Words that appear in similar contexts will have vectors that are close to each other in the vector space. We measure this closeness using **Cosine Similarity**.

**Your Task:** Use `gensim`'s built-in functions to find the similarity between two words, and find the top most similar words to a target word.

In [ ]:

sim_india_delhi = w2v_model.wv.similarity('india', 'delhi')
print(f"Cosine similarity between 'india' and 'delhi': {sim_india_delhi:.4f}")


similar_to_beautiful = w2v_model.wv.most_similar('beautiful', topn=3)
print("\nWords most similar to 'beautiful':")
for word, score in similar_to_beautiful:
    print(f" - {word}: {score:.4f}")

#(Note: On a tiny toy corpus, these similarities might not make perfect logical sense, but the code works!)

## Task 4: Visualizing Embeddings with t-SNE (Medium)

Our word vectors have 20 dimensions. Humans can only see in 2D or 3D. 
**t-SNE (t-Distributed Stochastic Neighbor Embedding)** is a dimensionality reduction technique that squashes our 20-dimensional vectors into 2 dimensions so we can plot them on a scatter plot.

**Your Task:** Extract the word vectors, apply t-SNE, and plot them using Matplotlib.

In [ ]:
def plot_tsne(model):
    # 1. Extract words and their vectors
    words = list(model.wv.index_to_key)
    vectors = np.array([model.wv[word] for word in words])
    
    # 2. Initialize t-SNE (perplexity must be less than the number of samples)
    tsne = TSNE(n_components=2, random_state=42, perplexity=5, init='pca')
    
    # 3. Fit and transform the vectors into 2D
    vectors_2d = tsne.fit_transform(vectors)
    
    # 4. Plotting
    plt.figure(figsize=(10, 8))
    plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], edgecolors='k', c='skyblue')
    
    # Add labels to points
    for i, word in enumerate(words):
        plt.annotate(word, xy=(vectors_2d[i, 0], vectors_2d[i, 1]), xytext=(5, 2), 
                     textcoords='offset points', ha='right', va='bottom', fontsize=9)
        
    plt.title("t-SNE Visualization of Custom Word2Vec Embeddings")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()


plot_tsne(w2v_model)

## Task 5: Comparing with Pre-trained GloVe Embeddings (Medium)

Training Word2Vec from scratch requires massive amounts of text (Wikipedia, news articles) to get good semantic representations. 

Instead of training from scratch, we can load **Pre-trained Embeddings** like GloVe (Global Vectors for Word Representation), which have already been trained on billions of words.

**Your Task:** Download a lightweight pre-trained GloVe model using `gensim` and perform a classic word analogy task: `King - Man + Woman = ?`

In [ ]:

print("Downloading GloVe model... Please wait.")
glove_model = api.load("glove-twitter-25")
print("Model loaded successfully!\n")


print("1. GloVe Similarity ('india', 'delhi'):", glove_model.similarity('india', 'delhi'))


analogy_result = glove_model.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)

print("\n2. Solving Analogy: [King - Man + Woman]")
print(f"Result: {analogy_result[0][0]} (Confidence: {analogy_result[0][1]:.4f})")

# Try an India-specific analogy: Mumbai is to Maharashtra as Delhi is to ?
# [Mumbai - Maharashtra + Delhi]
india_analogy = glove_model.most_similar(positive=['delhi', 'maharashtra'], negative=['mumbai'], topn=1)

print("\n3. Solving Analogy: [Delhi - Mumbai + Maharashtra]")
print(f"Result: {india_analogy[0][0]} (Confidence: {india_analogy[0][1]:.4f})")